In [ ]:
"""
YEAR + ONE COLUMN EXPORTER

What this does:
- Reads your two big CSV files from Downloads.
- Creates a new folder in Downloads:
    ~/Downloads/year_column_output/
- For every column in each file, creates one CSV file.
- Each output CSV contains only:
    year, selected_column
- Output is grouped/sorted by year.
- Original data files are NOT changed.

Example output:
~/Downloads/year_column_output/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6/year/column_001_unitid.csv
~/Downloads/year_column_output/business_startup_ALL_IN_ONE/year/column_001_source.csv
"""

from pathlib import Path
import csv
import re
import shutil
import pandas as pd


# ============================================================
# YOUR INPUT FILES
# ============================================================
DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

# Output goes to Downloads, not inside the original data folder
OUTPUT_ROOT = Path.home() / "Downloads/year_column_output"

# Good memory-safe default
CHUNKSIZE = 100_000

# False = keep blank cells as blank.
# True = remove rows where the selected column is blank.
DROP_EMPTY_COLUMN_VALUES = False

# None means export all columns.
# To test first, you can change this to 5.
MAX_COLUMNS_PER_FILE = None


# ============================================================
# HELPER FUNCTIONS
# ============================================================
def clean_filename(name):
    """Make a safe filename from a column name."""
    name = str(name).strip()
    if not name:
        name = "blank_column_name"
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name[:80] if name else "blank_column_name"


def read_header(path):
    """Read the CSV header exactly, including duplicate names."""
    with open(path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
        reader = csv.reader(f)
        return next(reader)


def make_unique_column_names(columns):
    """
    Pandas needs unique column names.
    This keeps the first column name as-is, then adds .1, .2, etc. for duplicates.
    """
    seen = {}
    result = []
    for col in columns:
        col = str(col)
        if col not in seen:
            seen[col] = 0
            result.append(col)
        else:
            seen[col] += 1
            result.append(f"{col}.{seen[col]}")
    return result


def find_year_column_index(columns):
    """Find the year column using exact lowercase match."""
    for i, col in enumerate(columns):
        if str(col).strip().lower() == "year":
            return i
    raise ValueError("No year column found in this file.")


def print_directory_tree(root):
    """Print the output directory tree."""
    root = Path(root)
    print()
    print("=" * 100)
    print("DOWNLOADS OUTPUT FOLDER DIRECTORY")
    print("=" * 100)
    print(root)

    if not root.exists():
        print("(folder does not exist yet)")
        return

    def walk(folder, prefix=""):
        entries = sorted(folder.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for index, entry in enumerate(entries):
            connector = "└── " if index == len(entries) - 1 else "├── "
            print(prefix + connector + entry.name)
            if entry.is_dir():
                extension = "    " if index == len(entries) - 1 else "│   "
                walk(entry, prefix + extension)

    walk(root)


def safe_read_csv_chunks(path, usecols, names, chunksize):
    """
    Read a CSV in chunks.
    usecols uses column positions, so duplicate column names are okay.
    """
    kwargs = dict(
        filepath_or_buffer=path,
        usecols=usecols,
        names=names,
        header=0,
        chunksize=chunksize,
        dtype=str,
        low_memory=False,
        keep_default_na=False,
        na_values=[],
        encoding="utf-8-sig",
    )

    try:
        return pd.read_csv(**kwargs, encoding_errors="replace")
    except TypeError:
        # Older pandas may not support encoding_errors
        return pd.read_csv(**kwargs)


def append_csv(path, df):
    """Append a dataframe to CSV and write header only once."""
    path = Path(path)
    write_header = not path.exists()
    df.to_csv(path, mode="a", index=False, header=write_header)


def combine_year_temp_files(temp_dir, final_path, years_seen):
    """Combine temporary year files into one final CSV sorted by year."""
    final_path = Path(final_path)
    first_file = True

    with open(final_path, "w", encoding="utf-8", newline="") as out_f:
        for year in sorted(years_seen):
            temp_file = temp_dir / f"year_{year}.csv"
            if not temp_file.exists():
                continue

            with open(temp_file, "r", encoding="utf-8", newline="") as in_f:
                for line_number, line in enumerate(in_f):
                    # Keep header only from the first temp file
                    if not first_file and line_number == 0:
                        continue
                    out_f.write(line)

            first_file = False


def export_one_column(source_path, output_dir, original_columns, unique_columns, year_idx, col_idx):
    """
    Export one selected column + year.
    Uses temporary files by year so the final output is sorted by year.
    """
    original_col_name = original_columns[col_idx]
    internal_year_name = unique_columns[year_idx]
    internal_col_name = unique_columns[col_idx]

    output_col_name = original_col_name
    if str(output_col_name).strip().lower() == "year":
        output_col_name = "year_value"

    safe_col_name = clean_filename(original_col_name)
    final_file = output_dir / f"column_{col_idx + 1:03d}_{safe_col_name}.csv"

    temp_dir = output_dir / "_temp_sort_by_year" / f"column_{col_idx + 1:03d}_{safe_col_name}"
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    temp_dir.mkdir(parents=True, exist_ok=True)

    usecols = sorted(set([year_idx, col_idx]))
    years_seen = set()
    rows_written = 0

    reader = safe_read_csv_chunks(
        path=source_path,
        usecols=usecols,
        names=unique_columns,
        chunksize=CHUNKSIZE,
    )

    for chunk in reader:
        # Convert year to integer where possible
        year_numeric = pd.to_numeric(chunk[internal_year_name], errors="coerce")

        if col_idx == year_idx:
            selected_values = chunk[internal_year_name]
        else:
            selected_values = chunk[internal_col_name]

        out = pd.DataFrame({
            "year": year_numeric,
            output_col_name: selected_values,
        })

        out = out.dropna(subset=["year"])
        out["year"] = out["year"].astype(int)

        if DROP_EMPTY_COLUMN_VALUES:
            out = out[out[output_col_name].astype(str).str.strip() != ""]

        if out.empty:
            continue

        # Write each year group to a temporary year file
        for year, group in out.groupby("year", sort=True):
            year = int(year)
            temp_file = temp_dir / f"year_{year}.csv"
            append_csv(temp_file, group)
            years_seen.add(year)
            rows_written += len(group)

    if years_seen:
        combine_year_temp_files(temp_dir, final_file, years_seen)
    else:
        # Write an empty file with only the header
        pd.DataFrame(columns=["year", output_col_name]).to_csv(final_file, index=False)

    shutil.rmtree(temp_dir, ignore_errors=True)

    return final_file, rows_written, sorted(years_seen)


def export_file(source_path):
    """Export all columns from one source file."""
    source_path = Path(source_path)

    print()
    print("=" * 100)
    print(f"CHECKING FILE: {source_path}")
    print("=" * 100)

    if not source_path.exists():
        raise FileNotFoundError(f"Missing file: {source_path}")

    original_columns = read_header(source_path)
    unique_columns = make_unique_column_names(original_columns)
    year_idx = find_year_column_index(original_columns)

    file_output_dir = OUTPUT_ROOT / source_path.stem / "year"
    file_output_dir.mkdir(parents=True, exist_ok=True)

    print(f"FOUND FILE: {source_path}")
    print(f"TOTAL COLUMNS: {len(original_columns)}")
    print(f"YEAR COLUMN: column_{year_idx + 1:03d}_{original_columns[year_idx]}")
    print(f"OUTPUT FOLDER: {file_output_dir}")

    columns_to_export = list(range(len(original_columns)))
    if MAX_COLUMNS_PER_FILE is not None:
        columns_to_export = columns_to_export[:MAX_COLUMNS_PER_FILE]

    summary_rows = []

    for n, col_idx in enumerate(columns_to_export, start=1):
        col_name = original_columns[col_idx]
        print()
        print(f"[{n}/{len(columns_to_export)}] Exporting column_{col_idx + 1:03d}_{col_name}")

        final_file, rows_written, years_seen = export_one_column(
            source_path=source_path,
            output_dir=file_output_dir,
            original_columns=original_columns,
            unique_columns=unique_columns,
            year_idx=year_idx,
            col_idx=col_idx,
        )

        summary_rows.append({
            "column_number": col_idx + 1,
            "column_name": col_name,
            "output_file": str(final_file),
            "rows_written": rows_written,
            "first_year": min(years_seen) if years_seen else "",
            "last_year": max(years_seen) if years_seen else "",
            "year_count": len(years_seen),
        })

        print(f"Saved: {final_file}")
        print(f"Rows: {rows_written:,}")

    summary = pd.DataFrame(summary_rows)
    summary_file = OUTPUT_ROOT / source_path.stem / "export_summary.csv"
    summary.to_csv(summary_file, index=False)

    print()
    print(f"SUMMARY SAVED: {summary_file}")

    return summary_file


def main():
    print("=" * 100)
    print("YEAR + ONE COLUMN EXPORTER")
    print("=" * 100)
    print("This will create one CSV per column.")
    print("Each CSV will contain only: year + that one column.")
    print("Original files will not be changed.")
    print(f"Output folder: {OUTPUT_ROOT}")

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    export_file(DEGREE_FILE)
    export_file(COMPANY_FILE)

    # Remove temp folder if it still exists
    temp_root = OUTPUT_ROOT / "_temp_sort_by_year"
    shutil.rmtree(temp_root, ignore_errors=True)

    print_directory_tree(OUTPUT_ROOT)

    print()
    print("=" * 100)
    print("DONE")
    print("=" * 100)
    print(f"Open this folder in Finder:")
    print(OUTPUT_ROOT)


if __name__ == "__main__":
    main()